# Training and Evaluation for SVM Regressor (RBF Kernel) 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## add more imports as needed.

In [4]:
# Load in all training datasets + water quality dataset

water_quality_df = pd.read_csv("../data/water_quality_training_dataset.csv")
terraclimate_df = pd.read_csv("../data/terraclimate_features_training.csv")
nasadem_df = pd.read_csv("../data/nasadem_features_training.csv")
modisVI_df = pd.read_csv("../data/modisVI_features_training.csv")
modisLST_df = pd.read_csv("../data/modisLST_features_training.csv")
landsat_df = pd.read_csv("../data/landsat_features_training.csv")
jrc_df = pd.read_csv("../data/jrc_features_training.csv")
isda_df = pd.read_csv("../data/iSDA_features_training.csv")
hydrosheds_df = pd.read_csv("../data/hydrosheds_features_training.csv")
era5_df = pd.read_csv("../data/era5_features_training.csv")
chirps_df = pd.read_csv("../data/chirps_features_training.csv")

In [5]:
## Join all features into a single Pandas DataFrame. (function from Benchmark Notebook)

# Combine two datasets vertically (along columns) using pandas concat function.
def combine_ten_datasets(d1,d2,d3,d4, d5, d6, d7, d8, d9, d10, d11):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([d1,d2,d3, d4, d5, d6, d7, d8, d9, d10, d11], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [6]:
wq_df = combine_ten_datasets(water_quality_df,
                             terraclimate_df,
                             nasadem_df,
                             modisVI_df,
                             modisLST_df,
                             landsat_df,
                             jrc_df,
                             isda_df, 
                             hydrosheds_df,
                             era5_df,
                             chirps_df)

wq_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,Unnamed: 0,elevation,EVI_median,...,phosphorous_mean,phosphorous_median,flow_accumulation,skin_temperature_median,soil_temperature_level_1_median,temperature_2m_median,total_evaporation_sum_median,total_precipitation_sum_median,volumetric_soil_water_layer_1_median,precipitation
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,174.2,0,167.155040,241.0,...,22.926434,23.0,4.131443e+06,307.919884,307.833471,300.198610,-0.000085,0.000001,0.008799,0.488024
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,124.1,1,1521.251493,4643.0,...,21.057342,21.0,1.162031e+04,293.665109,294.393847,292.444443,-0.004103,0.007830,0.458929,97.342756
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,127.5,2,1471.379902,3984.0,...,20.680648,21.0,1.000000e+00,293.972931,294.725525,292.828033,-0.004010,0.006964,0.393498,96.911494
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,129.7,3,1342.659998,2217.0,...,22.007566,22.0,1.878500e+04,294.280160,295.166729,293.509254,-0.003895,0.007228,0.409018,102.307634
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,129.2,4,1355.983661,4136.0,...,20.045393,20.0,4.778000e+03,294.414705,295.304085,293.890963,-0.003909,0.004980,0.437815,96.513882


In [8]:
wq_df.columns

Index(['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'pet',
       'Unnamed: 0', 'elevation', 'EVI_median', 'NDVI_median',
       'Land Surface Temperature', 'nir', 'green', 'swir16', 'swir22', 'NDMI',
       'MNDWI', 'occurrence', 'seasonality', 'cec_mean', 'cec_median',
       'clay_mean', 'clay_median', 'pH_mean', 'pH_median', 'phosphorous_mean',
       'phosphorous_median', 'flow_accumulation', 'skin_temperature_median',
       'soil_temperature_level_1_median', 'temperature_2m_median',
       'total_evaporation_sum_median', 'total_precipitation_sum_median',
       'volumetric_soil_water_layer_1_median', 'precipitation'],
      dtype='object')

In [9]:
# Drop irrelevant columns and mean columns for data that has the same but with median since median is more robust to outliers.

wq_df = wq_df.drop(columns=['Unnamed: 0', 'cec_mean', 'clay_mean', 'pH_mean', 'phosphorous_mean'])
wq_df.columns

Index(['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'pet',
       'elevation', 'EVI_median', 'NDVI_median', 'Land Surface Temperature',
       'nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'occurrence',
       'seasonality', 'cec_median', 'clay_median', 'pH_median',
       'phosphorous_median', 'flow_accumulation', 'skin_temperature_median',
       'soil_temperature_level_1_median', 'temperature_2m_median',
       'total_evaporation_sum_median', 'total_precipitation_sum_median',
       'volumetric_soil_water_layer_1_median', 'precipitation'],
      dtype='object')

In [11]:
## remove _median ending from columns by renaming

wq_df = wq_df.rename(columns={'EVI_median': 'EVI',
                      'NDVI_median': 'NDVI',
                      'cec_median': 'cec',
                      'clay_median': 'clay',
                      'pH_median': 'pH',
                      'phosphorous_median': 'phosphorous',
                      'skin_temperature_median': 'skin_temperature',
                      'soil_temperature_level_1_median': 'soil_temperature',
                      'temperature_2m_median': 'temperature_2m',
                      'total_evaporation_sum_median': 'total_evaporation_sum',
                      'total_precipitation_sum_median': 'total_precipitation_sum',
                      'volumetric_soil_water_layer_1_median': 'volumetric_soil_water',
                      })

wq_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,...,pH,phosphorous,flow_accumulation,skin_temperature,soil_temperature,temperature_2m,total_evaporation_sum,total_precipitation_sum,volumetric_soil_water,precipitation
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,...,75.0,23.0,4.131443e+06,307.919884,307.833471,300.198610,-0.000085,0.000001,0.008799,0.488024
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,...,65.0,21.0,1.162031e+04,293.665109,294.393847,292.444443,-0.004103,0.007830,0.458929,97.342756
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,...,64.0,21.0,1.000000e+00,293.972931,294.725525,292.828033,-0.004010,0.006964,0.393498,96.911494
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,...,65.0,22.0,1.878500e+04,294.280160,295.166729,293.509254,-0.003895,0.007228,0.409018,102.307634
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,...,64.0,20.0,4.778000e+03,294.414705,295.304085,293.890963,-0.003909,0.004980,0.437815,96.513882


### Exploratory Data Analysis

Things to check:
+ data skew
+ range
+ correlation heatmap
+ histograms